In [46]:
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

import lsst.geom
from lsst.daf.butler import Butler
from lsst.rsp import get_tap_service

In [47]:
rsp_tap = get_tap_service("tap")
assert rsp_tap is not None

In [48]:
butler = Butler('dp1', collections='LSSTComCam/DP1')
assert butler is not None

In [ ]:
# SELECTION = 1 chooses less distorted PSF data and SELECTION = 2 chooses more distorted PSF data
SELECTION = 2

if(SELECTION == 1):
    ra, dec = 106.3, -10.40
    my_band = 'r'
elif(SELECTION == 2):
    ra, dec = 53.13, -28.10
    my_band = 'i'

In [50]:
# Step 1: Query Butler for a visit_image by sky position and band
visitimage_refs = butler.query_datasets(
        'visit_image',
        where="visit.region OVERLAPS POINT(ra, dec) "
              f"and band='{my_band}'",
        bind={'ra': ra, 'dec': dec},
        order_by=['visit', 'detector'],
        with_dimension_records=True,
        limit=1
)

print(f"Found {len(visitimage_refs)} visit_image reference(s)")
print(visitimage_refs)

Found 1 visit_image reference(s)
[DatasetRef(DatasetType('visit_image', {band, instrument, day_obs, detector, physical_filter, visit}, ExposureF), {instrument: 'LSSTComCam', detector: 0, visit: 2024120200214, band: 'r', day_obs: 20241202, physical_filter: 'r_03'}, run='LSSTComCam/runs/DRP/DP1/DM-53601', id=019b1949-bf82-75d8-9788-83e1233c1d1f)]


In [51]:
# Step 2: Retrieve the visit_image and the local Rubin PSF object
visit_id = visitimage_refs[0].dataId['visit']
detector_id = visitimage_refs[0].dataId['detector']
print(f"Visit: {visit_id}, Detector: {detector_id}")

visit_image = butler.get(visitimage_refs[0])
psf = visit_image.getPsf()

reference_position = lsst.geom.Point2D(300, 300)
reference_stamp_dims = psf.computeImage(reference_position).getDimensions()

print('Route A reference chain: Butler -> visit_image -> getPsf() -> computeImage(position) -> psf_array')
print(f"Local Rubin PSF object retrieved. Reference stamp size: {reference_stamp_dims} pix")


Visit: 2024120200214, Detector: 0
Route A reference chain: Butler -> visit_image -> getPsf() -> computeImage(position) -> psf_array
Local Rubin PSF object retrieved. Reference stamp size: (25, 25) pix


In [52]:
# Step 3: Query dp1.Source for stars used in PSF modeling (calib_psf_used = 1)
query = (
    "SELECT sourceId, x, y "
    "FROM dp1.Source "
    f"WHERE visit = {visit_id} "
    f"AND detector = {detector_id} "
    "AND calib_psf_used = 1"
)

job = rsp_tap.submit_job(query)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()

psf_used_table = job.fetch_result().to_table()
print(f"Found {len(psf_used_table)} PSF candidate stars")

Job phase is COMPLETED
Found 287 PSF candidate stars


In [53]:
import sys
from pathlib import Path

for src_path in (Path.cwd() / 'src', Path.cwd().parent / 'src'):
    if src_path.exists() and str(src_path) not in sys.path:
        sys.path.insert(0, str(src_path))
        break

import importlib
import fittingTools
importlib.reload(fittingTools)


<module 'fittingTools' from '/Users/jinshuozhang/Library/CloudStorage/Dropbox/Rubin-Work/Rubin-PSF-Analysis/src/fittingTools.py'>

In [ ]:
# Step 4: Route B -- extract observed star cutouts and compare models

n_stars = len(psf_used_table)

print('Route B analysis chain: Butler -> visit_image -> getCutout(position, stamp size) -> observed star array')

t0 = time.time()
results = []
n_skipped = 0
for i in range(n_stars):
    try:
        x_star = psf_used_table['x'][i]
        y_star = psf_used_table['y'][i]
        position_star = lsst.geom.Point2D(x_star, y_star)

        rubin_psf_model = psf.computeImage(position_star)
        stamp_dims = rubin_psf_model.getDimensions()
        xlen = stamp_dims.getX()
        ylen = stamp_dims.getY()

        star_cutout = visit_image.getCutout(position_star, lsst.geom.Extent2I(xlen, ylen))
        star_masked_image = star_cutout.getMaskedImage()
        star_image_array = star_masked_image.image.array.astype(float)

        star_flux = np.sum(star_image_array)
        if not np.isfinite(star_flux) or star_flux <= 0:
            raise ValueError('non-positive observed cutout flux')

        star_image_array = star_image_array / star_flux

        psf_shape = psf.computeShape(position_star)
        psf_sigma = psf_shape.getDeterminantRadius()
        psf_fwhm = psf_sigma * fittingTools.SIGMA_TO_FWHM

        result = fittingTools.analyze_psf_models(
            star_image_array,
            x=x_star,
            y=y_star,
            fwhm=psf_fwhm,
        )
        result['rubin_psf_array'] = rubin_psf_model.array / np.sum(rubin_psf_model.array)
        results.append(result)

    except Exception as e:
        n_skipped += 1
        print('Star %d skipped: %s' % (i, e))

t_compute = time.time() - t0
print('Computation done: %d/%d stars in %.2fs (%d skipped)' % (
    len(results), n_stars, t_compute, n_skipped))


# --- VISUALIZATION PHASE ---
page_size = 3
n_computed = len(results)

t0 = time.time()
fittingTools.plot_model_comparison_pages(
    results,
    page_size,
    visit_id,
    detector_id,
    my_band,
)
best_model_counts = fittingTools.plot_best_model_counts(results)
t_plot = time.time() - t0
print('Plotting done: %d stars in %.2fs (%d per page)' % (n_computed, t_plot, page_size))
print('Best model counts:', best_model_counts)
